===========================================================================
# Generative AI Healthcare Translator - User Interface
===========================================================================

This notebook provides a simple user-facing interface for querying the healthcare narrative data developed in the project.

### What the user does
1. Select a healthcare domain.
2. Type a natural-language question.
3. Choose how many supporting records to retrieve.
4. Ask the project
5. Read the generated answer and inspect the supporting evidence.

The interface is designed to reuse the project's existing dataframes, embedding model, and generation pipeline whenever they are already loaded.


## Task 1: Imports and Setup
--------------------------------------------------------------------------


In [1]:
# Import libraries

import warnings
warnings.filterwarnings("ignore")

import html
import json
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer

print("Interface dependencies loaded.")

Interface dependencies loaded.


In [2]:
# Locate and load project outputs automatically

from pathlib import Path
import pandas as pd
import json

search_root = Path.cwd().parent

def find_project_file(filename):
    matches = list(search_root.rglob(filename))

    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {search_root}")

    return matches[0]


evidence_file = find_project_file("evidence_repository.csv")
prompt_file = find_project_file("prompt_experiment.csv")
testing_file = find_project_file("testing_results.csv")
communication_file = find_project_file("communication_prompts.json")

print("Evidence File:", evidence_file)
print("Prompt File:", prompt_file)
print("Testing File:", testing_file)
print("Communication Prompts:", communication_file)

evidence_repository = pd.read_csv(evidence_file)
prompt_experiment = pd.read_csv(prompt_file)
testing_results = pd.read_csv(testing_file)

with open(communication_file, "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

print("\nPROJECT DATA LOADED")
print("=" * 60)
print("Evidence Rows:", len(evidence_repository))
print("Prompt Experiment Rows:", len(prompt_experiment))
print("Testing Results:", len(testing_results))
print("Communication Styles:", list(communication_prompts.keys()))

Evidence File: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Project\Data\Narratives\evidence_repository.csv
Prompt File: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Project\Data\Narratives\prompt_experiment.csv
Testing File: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Project\Data\Evaluation\testing_results.csv
Communication Prompts: c:\Users\ahuyn\OneDrive\Documents\Education\MSDAE\Courses\IE7374\Project\Data\Narratives\communication_prompts.json

PROJECT DATA LOADED
Evidence Rows: 57298
Prompt Experiment Rows: 60
Testing Results: 20
Communication Styles: ['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']


## Task 2: Connect Project Data
--------------------------------------------------------------------------


In [3]:
# Task 2: Connect project data

narrative_data_dir = Path("Data/Narratives")
evaluation_data_dir = Path("Data/Evaluation")

evidence_file = narrative_data_dir / "evidence_repository.csv"
prompt_experiment_file = narrative_data_dir / "prompt_experiment.csv"
communication_prompts_file = narrative_data_dir / "communication_prompts.json"

required_files = {
    "Evidence Repository": evidence_file,
    "Prompt Experiment": prompt_experiment_file,
    "Communication Prompts": communication_prompts_file
}

missing_files = [f"{name}: {path}" for name, path in required_files.items() if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following required project files could not be found:\n\n" + "\n".join(missing_files)
    )

evidence_repository = pd.read_csv(evidence_file)
prompt_experiment = pd.read_csv(prompt_experiment_file)

with open(communication_prompts_file, "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

print("Evidence Rows:", len(evidence_repository))
print("Prompt Experiment Rows:", len(prompt_experiment))
print("Communication Styles:", list(communication_prompts.keys()))

FileNotFoundError: The following required project files could not be found:

Evidence Repository: Data\Narratives\evidence_repository.csv
Prompt Experiment: Data\Narratives\prompt_experiment.csv
Communication Prompts: Data\Narratives\communication_prompts.json

In [ ]:
# Double Check: Confirm application components are available

print("Generation Model:", generation_model_name)
print("Hospitals Available:", evidence_repository["Facility ID"].nunique())
print("Domains Available:", evidence_repository["Domain"].nunique())
print("Communication Styles:", len(communication_prompts))

print("\nAvailable Styles:")
print(list(communication_prompts.keys()))

In [ ]:
# Task 3: Create user interface controls

available_facilities = sorted(evidence_repository["Facility Name"].dropna().unique())
available_styles = list(communication_prompts.keys())

facility_dropdown = widgets.Dropdown(
    options=available_facilities,
    description="Hospital:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="95%")
)

style_dropdown = widgets.Dropdown(
    options=available_styles,
    value="Patient Friendly",
    description="Style:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="95%")
)

generate_button = widgets.Button(
    description="Generate Narrative",
    button_style="primary"
)

clear_button = widgets.Button(description="Clear")

status_html = widgets.HTML()
output_area = widgets.Output()

In [ ]:
# Task 4: Connect interface to hospital communication application

def handle_generate(_):

    output_area.clear_output()

    facility_name = facility_dropdown.value
    communication_style = style_dropdown.value

    status_html.value = "<i>Generating narrative...</i>"
    generate_button.disabled = True

    try:
        result = hospital_communication_app(
            facility_name=facility_name,
            communication_style=communication_style,
            prompt_version=1
        )

        with output_area:
            print("HOSPITAL PERFORMANCE COMMUNICATION")
            print("=" * 90)
            print("Hospital:", facility_name)
            print("Communication Style:", result["Communication Style"])
            print("Prompt Version:", result["Prompt Version"])

            print("\nPERFORMANCE EVIDENCE")
            print("-" * 90)
            print(result["Performance Evidence"])

            if result["Context Evidence"]:
                print("\nCONTEXT EVIDENCE")
                print("-" * 90)
                print(result["Context Evidence"])

            print("\nGENERATED NARRATIVE")
            print("-" * 90)
            print(result["Generated Narrative"])

        status_html.value = "<b>Generation complete.</b>"

    except Exception as exc:
        message = html.escape(f"{type(exc).__name__}: {exc}")
        status_html.value = (
            f"<b style='color:#b42318;'>Generation failed.</b><br><code>{message}</code>"
        )

    finally:
        generate_button.disabled = False


def handle_clear(_):
    status_html.value = ""
    output_area.clear_output()


generate_button.on_click(handle_generate)
clear_button.on_click(handle_clear)

In [ ]:
# Task 5: Display final user interface

header = widgets.HTML(
    """
    <h2>Hospital Performance Communication Assistant</h2>
    <p>
    Select a hospital and communication style to generate an audience-specific
    hospital performance narrative from the project's prepared evidence.
    </p>
    """
)

controls = widgets.VBox([
    facility_dropdown,
    style_dropdown,
    widgets.HBox([generate_button, clear_button]),
    status_html
])

interface = widgets.VBox([
    header,
    controls,
    widgets.HTML("<hr>"),
    output_area
])

display(interface)

## Task 6:  User interface
---------------

In [ ]:
available_domains = list(domain_data.keys())

domain_dropdown = widgets.Dropdown(
    options = available_domains if available_domains else ["No data available"],
    description = "Select:"
)


question_box = widgets.Textarea(
    value="",
    placeholder="Example: What themes appear in the patient narratives?",
    description="Question:",
    layout=widgets.Layout(width="95%", height="100px"),
    style={"description_width": "150px"},
)

top_k_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    step=1,
    description="Results",
    continuous_update=False,
)

ask_button = widgets.Button(
    description="Ask the Project",
    button_style="primary"
)

clear_button = widgets.Button(
    description="Clear"
)

status_html = widgets.HTML()
answer_output = widgets.Output()
evidence_output = widgets.Output()

def show_answer(answer, mode):
    with answer_output:
        clear_output()
        safe_answer = html.escape(answer).replace("\n", "<br>")
        card = f"""
        <h3>Generated Summary</h3>
        <p>{safe_answer}</p>
        <p><i>{html.escape(mode)}</i></p>
        
        """
        
        display(HTML(card))

def show_evidence(evidence):
    with evidence_output:
        clear_output()
        display(HTML("<h4>Supporting Evidence</h4>"))

        display_columns = ["Source_Row", "Similarity", TEXT_COLUMN]
        for col in evidence.columns:
            if col not in display_columns and len(display_columns) < 6:
                display_columns.append(col)

        display_df = evidence[display_columns].copy()
        display_df["Similarity"] = display_df["Similarity"].round(3)

        with pd.option_context("display.max_colwidth", 500, "display.max_rows", 20):
            display(display_df)

def handle_ask(_):
    answer_output.clear_output()
    evidence_output.clear_output()

    if not domain_data:
        status_html.value = (
            "<b style='color:#b42318;'>No datasets are connected. "
            "Run Section 3 after loading project data or setting CSV paths.</b>"
        )
        return

    question = question_box.value.strip()
    if not question:
        status_html.value = "<b style='color:#b42318;'>Please enter a question.</b>"
        return

    domain_name = domain_dropdown.value
    status_html.value = "<i>Searching evidence, please wait (may take a few minutes)"
    ask_button.disabled = True

    try:
        answer, evidence, mode = answer_question(
            domain_name,
            question,
            top_k_slider.value
        )
        show_answer(answer, mode)
        show_evidence(evidence)
        status_html.value = (
            f"<b>Complete.</b> Retrieved {len(evidence)} supporting records "
            f"from <i>{html.escape(domain_name)}</i>."
        )
    except Exception as exc:
        status_html.value = (
            "<b style='color:#b42318;'>The query could not be completed.</b><br>"
            f"<code>{html.escape(type(exc).__name__ + ': ' + str(exc))}</code>"
        )
    finally:
        ask_button.disabled = False

def handle_clear(_):
    question_box.value = ""
    status_html.value = ""
    answer_output.clear_output()
    evidence_output.clear_output()

ask_button.on_click(handle_ask)
clear_button.on_click(handle_clear)

header = widgets.HTML(
    """
    <h2>Healthcare Data Explorer</h2>
    <p>Ask a question about the hospital quality data and review the supporting evidence.</p>
    """
)

controls = widgets.VBox([
    domain_dropdown,
    question_box,
    top_k_slider,
    widgets.HBox([ask_button, clear_button]),
    status_html,
])

interface = widgets.VBox([
    controls,
    widgets.HTML("<hr>"),
    answer_output,
    evidence_output,
])

display(interface)

## Example Questions
Test questions such as:

- **Patient Survey:** What common concerns or positive experiences appear in patient narratives?
- **Healthcare-Associated Infections:** What infection-related patterns are represented in the retrieved records?
- **Timely and Effective Care:** What themes are associated with delays or effective treatment?
- What evidence is most relevant to a specific quality-of-care concern?
- What do the retrieved narratives suggest about this topic?